# 06 — Native-Grid ELM Land Initial Condition Comparisons (`mosaic`)

**One branch per run**: set `EVAL_BRANCH` in the configuration cell to `"clim"`, `"obs"` (ERA5), or `"obs_era5land_nsidc"` (ERA5-Land + NOAA/NSIDC). Figures and tables go to `DIAG_ROOT/<EVAL_BRANCH>/`.


This notebook uses [`mosaic`](https://docs.e3sm.org/mosaic/) (`build_elm_descriptor` + `mosaic.polypcolor`) to plot **ELM** initial conditions directly on the 94,838 active `grid1d` land cells without 2D raster regridding.

All helper functions live in [`util/ic_eval.py`](util/ic_eval.py), and figures are saved to `/global/cfs/cdirs/e3sm/www/zhan391/E3SMv3_S2D/rtenso2026_ic/figure/`.

### Key ELM Fields Evaluated
1. **Snow water equivalent (`H2OSNO`, mm)**
2. **Surface soil water (`H2OSOI_LIQ + H2OSOI_ICE`, layers 1–3, ~0–9 cm, $\text{kg m}^{-2}$)**
3. **Subsurface soil water (`H2OSOI_LIQ + H2OSOI_ICE`, layers 4+, >9 cm, $\text{kg m}^{-2}$)**
4. **Surface soil temperature (`T_SOISNO`, layer 1, K)**
5. **Subsurface soil temperature (`T_SOISNO`, layers 4-10, depth-weighted over 0.09-2.89 m, K)**

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path("util").resolve()))
import importlib
import clim_restart
import ic_eval as ice

# Reload so edits to util/ take effect without restarting the kernel.
importlib.reload(clim_restart)
ice = importlib.reload(ice)
from clim_restart import comparison_points

ice.ensure_proj_data()
plt.rcParams["figure.dpi"] = 120

# Branch evaluated in this run: "clim", "obs" (ERA5), or "obs_era5land_nsidc".
# Figures and tables go to DIAG_ROOT/<EVAL_BRANCH>/.
EVAL_BRANCH = "obs_era5land_nsidc"

SRC_ROOT     = Path("/global/cfs/cdirs/e3sm/zhan391/v3.LR.S2D.ENSINT/BruteForce")
# Observation sets for the obs branches (as in 02_build_climatology_ic.ipynb);
# "label" names each set's observed anomaly in the maps.
ERA5_DIR     = Path("/global/cfs/projectdirs/m3522/cmip6/ERA5/e5.oper.an.sfc")
OBS_DATA     = Path("/global/cfs/cdirs/e3sm/zhan391/data")
OBS_SETS = {
    "obs": {
        "label": "ERA5",
        "root": {"elm": ERA5_DIR, "mpassi": ERA5_DIR},
        "fields": {"elm": ["sd", "swvl1", "swvl2-4", "stl1", "stl2-4"], "mpassi": ["ci"]},
    },
    "obs_era5land_nsidc": {
        "label": "ERA5-Land/NSIDC",
        "root": {"elm": OBS_DATA / "ERA5_LND", "mpassi": OBS_DATA / "NOAA_NSIDC"},
        "fields": {"elm": ["sd", "swvl1", "swvl2-4", "stl1", "stl2-4"], "mpassi": ["ci"]},
    },
}
BRANCHES     = [EVAL_BRANCH]
OBS_REFS     = {b: s["label"] for b, s in OBS_SETS.items() if b == EVAL_BRANCH}   # obs branch -> observed-anomaly label
OUT_ROOT     = Path("/global/cfs/cdirs/e3sm/S2S2D/RTENSO2026/initial_conditions")
CACHE_DIR    = OUT_ROOT / "cache"
DIAG_ROOT    = Path("/global/cfs/cdirs/e3sm/www/zhan391/E3SMv3_S2D/rtenso2026_ic") / EVAL_BRANCH
SAVE_DIAG    = True

TEST_DATES   = ["2015-02-01", "2015-05-01", "2015-08-01", "2015-11-01"]
EXAMPLE_DATE = "2015-02-01"
BOX          = (15.0, 30.0)   # as in 02 (from 01_choose_box_size.ipynb)
BOX_SHIFTS   = 4             # as in 02 (selection distance only)
FIRST_YEAR   = 1995          # candidate years FIRST_YEAR-2018, as in 02 (None = 1980)

plans, distances, selections = ice.load_test_selections(
    TEST_DATES, SRC_ROOT, OUT_ROOT, obs_sets=OBS_SETS, branches=BRANCHES, first_year=FIRST_YEAR,
    box=BOX, box_shifts=BOX_SHIFTS, cache_dir=CACHE_DIR
)
obs_plans_ex = {label: plans[b, EXAMPLE_DATE] for b, label in OBS_REFS.items() if (b, EXAMPLE_DATE) in plans}

pts_elm = comparison_points(plans[EVAL_BRANCH, EXAMPLE_DATE], "elm", CACHE_DIR)
elm_desc = ice.build_elm_descriptor(pts_elm["lat"], pts_elm["lon"])
print(f"Built ELM mosaic.Descriptor for {pts_elm['lat'].size:,} native grid1d land cells")

## 1. State and Difference Maps across Four Seasons

> **Figure 1 Caption.** Evaluation of native-grid ELM initial condition states and relative errors across five key prognostic land variables (rows 1–5: snow water equivalent [mm], surface soil water [kg m$^{-2}$], subsurface soil water [kg m$^{-2}$], surface soil temperature [K], and subsurface soil temperature [K]) on 94,838 active `grid1d` land cells (Robinson projection). For each seasonal date (`TEST_DATES`: `2015-02-01`, `2015-05-01`, `2015-08-01`, `2015-11-01`), the four columns display:
> 1. **Baseline (Col 1):** E3SM 1995–2018 candidate-year climatological mean state for the calendar date, representing an unconstrained baseline without year selection.
> 2. **Reference (Col 2):** Actual E3SM 2015 state (the true simulation restart target).
> 3. **Selected Initial Condition (Col 3):** E3SM state from the candidate year selected by anomaly matching against ERA5-Land observations (`EVAL_BRANCH`). In-panel metrics (PCC and RMSE) compare candidate and baseline states directly against the 2015 reference.
> 4. **Error (Col 4):** Percent difference relative to reference ($100 \times [\text{selected} - \text{ref}] / \text{ref}$). Titles report domain relative bias (%) and pattern correlation on 15°×30° lat-lon boxes (`box PCC`).
>
> Full-field colorbars span Columns 1–3 with fine variable-specific binning; error maps (Column 4) use symmetric diverging scales centered on neutral gray ($|\Delta| \approx 0$).


In [ ]:
# --- Figure setup ---
FONTSIZE = 14        # panel-title font size; titles, labels, and ticks scale from it
PANEL_SIZE = None    # (width, height) of each panel in inches; None scales with FONTSIZE
DPI = 150            # resolution of the saved PNG
DATES = TEST_DATES   # ['2015-02-01', '2015-05-01', '2015-08-01', '2015-11-01']

elm_fields = [
    "snow water equivalent",
    "surface soil water",
    "subsurface soil water",
    "surface soil temperature",
    "subsurface soil temperature",
]

spatial_elm_by_date = {}

for date in DATES:
    plan_d = plans[EVAL_BRANCH, date]
    byears_d = {b: selections[b, date]["elm"] for b in BRANCHES if (b, date) in selections}
    obs_plans_d = {label: plans[b, date] for b, label in OBS_REFS.items() if (b, date) in plans}

    spatial_elm = {
        fname: ice.prepare_spatial_field(
            plan_d, "elm", fname, 2015, byears_d, obs_plans=obs_plans_d, box=BOX, cache_dir=CACHE_DIR,
            log=lambda *_: None,
        )
        for fname in elm_fields
    }
    spatial_elm_by_date[date] = spatial_elm

    # All five fields in one figure, one row each.
    fig = ice.plot_elm_spatial([spatial_elm[f] for f in elm_fields], elm_descriptor=elm_desc, kind="state_and_diff",
                               fontsize=FONTSIZE, panel_size=PANEL_SIZE)
    ice.save_fig(fig, f"elm_state_diff_{date.replace('-', '')}", DIAG_ROOT, SAVE_DIAG, dpi=DPI)
    plt.show()

# Retain EXAMPLE_DATE spatial dict for downstream cells
spatial_elm = spatial_elm_by_date.get(EXAMPLE_DATE, spatial_elm)

## 2. Standardized Anomaly ($z$) Maps across Four Seasons

> **Figure 2 Caption.** Spatial patterns of standardized anomalies ($z = [x - \bar{x}] / \sigma_x$) for the five key ELM land variables (rows 1–5), plotted directly on native `grid1d` land mesh cells. For each seasonal target date (`TEST_DATES`: `2015-02-01`, `2015-05-01`, `2015-08-01`, `2015-11-01`), panels compare:
> 1. **Truth (Col 1):** Observed anomaly on the initialization date derived from ERA5-Land reanalysis relative to its own 1995–2018 climatology.
> 2. **Reference (Col 2):** Actual E3SM 2015 standardized anomaly.
> 3. **Selected Initial Condition (Col 3):** Standardized anomaly of the E3SM candidate year chosen by observation matching.
>
> **In-Panel Metrics:** Grid-point pattern correlation (`PCC`), grid-point root-mean-square error (`RMSE` in $\sigma$ units), and 15°×30° box pattern correlation (`box PCC`), each evaluated against the observed truth (Column 1).
>
> **Colorbar:** Standardized anomaly scale spanning $[-3\sigma, +3\sigma]$ with warm hues (red) indicating positive anomalies, cool hues (blue) indicating negative anomalies, and an inner neutral gray band ($|z| < 0.5\sigma$) highlighting significant departures.


In [ ]:
# --- Figure setup ---
FONTSIZE = 16        # panel-title font size; titles, labels, and ticks scale from it
PANEL_SIZE = None    # (width, height) of each panel in inches; None scales with FONTSIZE
DPI = 150            # resolution of the saved PNG
Z_LIM = 3.0          # color range of the standardized anomaly (z) maps
Z_LEVELS = [-3, -2, -1.5, -1, -0.5, 0, 0.5, 1, 1.5, 2, 3]  # nice round anomaly levels
DATES = TEST_DATES   # ['2015-02-01', '2015-05-01', '2015-08-01', '2015-11-01']

for date in DATES:
    sp_d = spatial_elm_by_date.get(date)
    if sp_d is None:
        plan_d = plans[EVAL_BRANCH, date]
        byears_d = {b: selections[b, date]["elm"] for b in BRANCHES if (b, date) in selections}
        obs_plans_d = {label: plans[b, date] for b, label in OBS_REFS.items() if (b, date) in plans}
        sp_d = {
            fname: ice.prepare_spatial_field(
                plan_d, "elm", fname, 2015, byears_d, obs_plans=obs_plans_d, box=BOX, cache_dir=CACHE_DIR,
                log=lambda *_: None,
            )
            for fname in elm_fields
        }

    # All five fields in one figure, one row each.
    fig = ice.plot_elm_spatial(
        [sp_d[f] for f in elm_fields], elm_descriptor=elm_desc, kind="z",
        z_lim=Z_LIM, z_levels=Z_LEVELS, fontsize=FONTSIZE, panel_size=PANEL_SIZE,
    )
    ice.save_fig(fig, f"elm_z_{date.replace('-', '')}", DIAG_ROOT, SAVE_DIAG, dpi=DPI)
    plt.show()


## 3. Four-Season ELM Anomaly Comparison (`2015-02-01` to `2015-11-01`)

> **Figure 3 Caption.** Seasonal progression of standardized anomalies ($z$) across all four 2015 target dates (rows: February 1, May 1, August 1, and November 1) for an individual ELM variable. Columns show the observed truth (ERA5-Land), actual E3SM 2015 state, and each evaluated candidate-year selection branch, illustrating the seasonal cycle of memory and anomaly retention in land reservoirs (snowpack, shallow soil moisture, root-zone moisture, and deep thermal inertia).


In [ ]:
# --- Figure setup ---
FONTSIZE = 14        # panel-title font size; titles, labels, and ticks scale from it
PANEL_SIZE = None    # (width, height) of each panel in inches; None scales with FONTSIZE
DPI = 150            # resolution of the saved PNG
Z_LIM = 2.5          # color range of the standardized anomaly (z) maps

for fname in ["snow water equivalent", "surface soil water", "subsurface soil water", "surface soil temperature",
              "subsurface soil temperature"]:
    fig = ice.plot_elm_four_seasons(
        fname, TEST_DATES, plans, selections, elm_desc,
        obs_refs=OBS_REFS, box=BOX, cache_dir=CACHE_DIR, z_lim=Z_LIM, fontsize=FONTSIZE, panel_size=PANEL_SIZE,
    )
    ice.save_fig(fig, f"elm_4seasons_z_{ice.slugify(fname)}_2015", DIAG_ROOT, SAVE_DIAG, dpi=DPI)
    plt.show()

## 4. Key findings

Evaluated branch: `obs_era5land_nsidc` (`EVAL_BRANCH`); example date 2015-02-01 (selected year 2014).

**Table. Land key fields on 2015-02-01.** Left: RMSE of the absolute state against the actual 2015 model state
(section 1). Right: pattern correlation (PCC, at grid points / on 15x30 boxes) of the standardized anomaly
against the observed ERA5-Land anomaly (section 2).

| Field | RMSE vs actual 2015: selected year | RMSE vs actual 2015: baseline (1995-2018 mean) | PCC vs observation: actual 2015 | PCC vs observation: selected year |
|---|---|---|---|---|
| snow water equivalent | **22.8 mm** | 27.7 mm | 0.32 / 0.79 | 0.16 / 0.46 |
| surface soil water | 5.12 kg m$^{-2}$ | **4.2** | 0.28 / 0.56 | 0.01 / 0.10 |
| subsurface soil water | 36.7 kg m$^{-2}$ | **34.7** | 0.40 / 0.65 | 0.19 / 0.31 |
| surface soil temperature | 3.54 K | **2.15** | 0.67 / 0.85 | -0.01 / -0.01 |
| subsurface soil temperature | 1.09 K | **0.89** | 0.53 / 0.69 | 0.21 / 0.37 |

1. **The selected year beats the climatological baseline only for snow.** Against the actual 2015 state, the
   1995-2018 mean has a lower RMSE than the selected year for soil water and soil temperature. A mean of 23
   years is smooth and hedges toward the middle, while any single year carries its own full-size anomalies;
   the selected year is still the best single, physically consistent year.
2. **The observed anomaly is closer to the actual 2015 state than the selected year is**, for every land field
   (box PCC 0.56-0.85 against 0.10-0.46 at box scale), most strongly for soil temperature: the observed
   surface soil-temperature anomaly matches actual 2015 well (box PCC 0.85), while the selected year has no
   resemblance (-0.01).
3. **Deeper soil temperature is matched better than surface soil temperature** by the selected year (box PCC
   0.37 vs -0.01), consistent with its slower variation.
4. **Implication.** Transferring the observed anomaly onto a climatological base (`clim_anom`,
   `03_build_clim_anomaly_ic.ipynb`) combines the strengths of both: the smoothness of the climatology and the
   observed anomaly that no single archive year reproduces.

These numbers come from the figures above; update this section if the configuration changes.